# DMIR × µNAS — Lane-Change Intention on a Microcontroller Budget

**Goal.** Beat the published baseline on the DMIR lane-change tasks (92% accuracy for 3-class intention; 0.42 s / 0.44 s RMSE for LCR / LCL time-to-lane-change) with architectures found by constrained NAS, small enough for STM32 deployment.

**How to use this notebook.**
1. Every code cell is short and calls functions from [`src/`](../src); long logic lives there.
2. **All knobs are in the Configuration cell (§2)** — edit only there, then *Run All*.
3. Each run appends a machine-readable record to `datasets/dmir/logs/experiments.jsonl` (feeds the paper) and can add a note to `LOGBOOK.md`.
4. After any change to `src/`, run `python scripts/check_pipeline.py` — a 3-task smoke test on real data.

Data facts and known quirks: [`datasets/dmir/docs/DATA.md`](../datasets/dmir/docs/DATA.md).

## 1 · Environment

Import the project package and report Python / PyTorch / GPU. Expect `cuda_available: True` on the RTX 5070 (CUDA 12.8 wheels).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.env_utils import env_report, pick_device, seed_everything

device = pick_device()
env_report()

## 2 · Configuration — edit here only

One `Config` object controls the whole pipeline. Key knobs:

| Field | Meaning |
|---|---|
| `task` | `classification`, `regression_lcl`, or `regression_lcr` |
| `subset_fraction` | `1.0` = full data; e.g. `0.05` for a quick smoke run |
| `clip_to_train_range` | clip val/test spikes to the train range (see `datasets/dmir/docs/DATA.md`) |
| `epochs`, `lr`, `batch_size`, `early_stop_patience` | training schedule |
| `run_name`, `notes` | how this run appears in `datasets/dmir/logs/experiments.jsonl` |

In [ ]:
from src.config import Config

cfg = Config(
    task="classification",      # classification | regression_lcl | regression_lcr
    subset_fraction=1.0,        # 1.0 = full dataset
    clip_to_train_range=True,
    epochs=60,
    batch_size=256,
    lr=3e-3,
    early_stop_patience=10,
    seed=42,
    run_name="baseline-dscnn",
    notes="depthwise-separable CNN baseline",
)
seed_everything(cfg.seed)
cfg

## 3 · Data loading

Loads the prepared pickles for the selected task (float32), clips val/test to the train range if enabled, and reports shapes plus how many cells were clipped.

In [ ]:
from src.data import load_task, make_loaders

bundle = load_task(cfg)
print(bundle.summary())

## 4 · EDA — label distribution

Classification splits are exactly balanced (1/3 per class). Regression targets cover 0.0–4.0 s in 0.1 s steps.

In [ ]:
from src import eda

eda.plot_label_distribution(bundle, cfg.is_classification);

## 4.1 · EDA — feature ranges

Per-feature min/max and spread on the training split. Note feature 7 (zero variance in the classification task) and the differing scales across features — the data arrives pre-normalised by the provider, so scales reflect their pipeline.

In [ ]:
eda.plot_feature_ranges(bundle);

## 4.2 · EDA — example windows

One random 50-timestep training window per class (or per target tercile for regression), showing a few features.

In [ ]:
eda.plot_example_windows(bundle, cfg.is_classification, features=(0, 1, 2, 8));

## 5 · Baseline model

Small depthwise-separable 1D CNN (MCU-friendly operator set). This is the internal reference point; NAS (§8) must beat it on the accuracy/footprint trade-off, and the final model must beat the published SOTA.

In [ ]:
from torchinfo import summary

from src.models import BaselineDSCNN

model = BaselineDSCNN(n_features=31, n_outputs=cfg.n_outputs)
summary(model, input_size=(1, 31, 50), depth=3)

## 6 · Training

AdamW + cosine schedule, mixed precision on GPU, early stopping on the validation metric (accuracy or RMSE), best weights restored. An `ExperimentLogger` starts recording here and is written to disk in §7.2.

In [ ]:
from src.log_utils import ExperimentLogger
from src.train import train_model

loaders = make_loaders(cfg, bundle)
logger = ExperimentLogger(cfg)
history = train_model(cfg, model, loaders, device)

## 6.1 · Training curves

Loss and validation metric per epoch; dashed line marks the restored best epoch.

In [ ]:
eda.plot_history(history, cfg.is_classification);

## 7 · Test evaluation vs. published SOTA

Targets to beat — classification: **92% accuracy**; regression RMSE: **0.42 s (LCR)** / **0.44 s (LCL)**. For classification a confusion matrix is shown; regression reports RMSE and MAE.

In [ ]:
from src.train import evaluate

test = evaluate(cfg, model, loaders["test"], device)
SOTA = {"classification": ("accuracy", 0.92, "higher"),
        "regression_lcl": ("rmse", 0.44, "lower"),
        "regression_lcr": ("rmse", 0.42, "lower")}
key, ref, direction = SOTA[cfg.task]
ours = test[key]
beat = ours > ref if direction == "higher" else ours < ref
print(f"test {key}: {ours:.4f}   (SOTA {ref}, {'BEATEN' if beat else 'not beaten yet'})")
if cfg.is_classification:
    print(f"macro-F1: {test['macro_f1']:.4f}   per-class recall: {test['per_class_recall']}")
    eda.plot_confusion_matrix(test["confusion_matrix"]);
else:
    print(f"MAE: {test['mae']:.4f} s")

## 7.1 · Persist the experiment record

Appends this run (config + environment + metrics + duration) to `datasets/dmir/logs/experiments.jsonl`. Set `WRITE_LOGBOOK_NOTE = True` to also add a human-readable entry to `LOGBOOK.md`.

In [ ]:
from src.log_utils import append_logbook

logger.log_metrics(best_epoch=history["best_epoch"],
                   best_val_metric=history["best_val_metric"],
                   train_time_s=history["train_time_s"],
                   **{f"test_{k}": v for k, v in test.items() if k != "confusion_matrix"})
path = logger.finish()
print(f"logged -> {path}")

WRITE_LOGBOOK_NOTE = False   # set True to journal this run
if WRITE_LOGBOOK_NOTE:
    append_logbook(f"Run {cfg.run_name} ({cfg.task})",
                   f"test {key} = {ours:.4f} (SOTA {ref}). Notes: {cfg.notes}")

## 8 · Constrained NAS (µNAS) — what the search found

Aging evolution over an MCU-safe 1D-CNN search space (ELIOS fork of µNAS; notes in [`unas/README.md`](../unas/README.md)). Fitness is the normalized-worst-violation over four objectives — validation error, peak RAM, flash (int8 params), MACs — with budgets set from the **STM32H7B3I-DK**. Four searches (classification, classification-without-indicators, LCR, LCL) ran **150 rounds each (population 50, sample 15)** and completed in about one GPU-night total on the RTX 5070 (LOGBOOK 2026-07-08/09).

The cells below **read the verified front CSVs** from [`datasets/dmir/results/nas-fronts/`](../datasets/dmir/results/nas-fronts) — every model was independently re-evaluated on the test split by `unas/harvest_fronts.py`; nothing here is retrained or simulated.

In [ ]:
from src import results_viz as rv

fronts = rv.load_fronts()
rv.front_summary(fronts)

## 8.1 · Classification fronts — the whole trade-off, one search

Faded dots are all saved candidates; the step line is the true non-dominated frontier (points saved as Pareto-optimal *at save time* can be dominated by later candidates). **`cls_best` (92.08% @ 84 k) and `cls_tiny` (91.30% @ 8 k) are two members of the same 33-model front** — the tiny model is a different architecture the search found, not a pruned copy of the best one. The green ablation series (no turn-indicator features) shows the model does not depend on the blinker: ~1 point below, everywhere on the size axis.

In [ ]:
rv.plot_classification_front(fronts);

## 8.2 · Regression fronts — against two baselines

Each dot is a searched model (MAE- and RMSE-objective runs combined per direction). Two reference lines per direction: the **published record** (single combined TTLC, RMSE 0.510 — beaten by both LCR 0.447 and LCL 0.466) and the **internal reference transformers** (0.42 / 0.44 — *not yet beaten*; different train/threshold protocol, so a soft comparison in both directions).

In [ ]:
rv.plot_regression_fronts(fronts);

## 9 · Quantization — actual artifact sizes on disk

Every deployment `.tflite` in [`datasets/dmir/results/deploy/`](../datasets/dmir/results/deploy), sizes read from disk at run time (KB). Three builds per model: float32, int16×8, full int8. int8 shrinks flash ~3× versus float32.

In [ ]:
rv.tflite_sizes()

## 9.1 · What int8 costs on this data

float32 vs full-int8 PTQ, our own test-set evaluation of each deployment artifact. The DMIR inputs have very different per-channel scales, so full-int8 costs real quality (classification −5 points, LCR MAE 0.287 → 0.449). int16×8 preserves accuracy offline — but it is **not deployable**: ST Edge AI 4.0.1 silently dequantizes it back to float32 (hardware-verified 2026-07-14). The two honest operating points are **float32 (accuracy)** and **int8 (size/speed)**.

In [ ]:
rv.plot_quantization_effect();

## 10 · Measured on real hardware — ST Edge AI Developer Cloud

On-device numbers, not estimates: **ST Edge AI Core 4.0.1-20581**, optimization *balanced*, board **STM32H7B3I-DK** (Cortex-M7 @ 280 MHz, 1184 KB RAM, 2048 KB flash), measured 2026-07-13/14 and recorded in [`datasets/dmir/docs/deployment.md`](../datasets/dmir/docs/deployment.md). The board matches the published baseline's deployment platform, so the comparison is like-for-like. It is also the unit available in the ELIOS lab.

Note the `cls_best int16x8` artifact is **absent** here by design: ST silently dequantizes int16×8 to float32 (its measured weights equal the float32 build byte-for-byte), so it is not a real deployment option; the empirical story is in lesson 09 of the course and in `deployment.md`.

In [ ]:
rv.MEASURED_H7B3

## 10.1 · Latency and flash, side by side

Latency on a log scale (spans 0.79 → 33.5 ms). Gray = the published reference CNN, blue = searched models. Headline, same board/settings: `cls_best` is **+0.4 accuracy points and 9.2× faster** than the reference; `cls_tiny` is **42× faster in 37 KB** at −0.4 points.

In [ ]:
rv.plot_measured_deployment();

## 10.2 · The categorical result — the smallest board

NUCLEO-F401RE (Cortex-M4 @ 84 MHz, 512 KB flash, 96 KB RAM). **The reference CNN cannot run on this board at all** — its flash need is 3.38× the entire chip; no toolchain setting fixes that. Every searched model fits, and `cls_tiny` was measured there: **4.376 ms** (2026-07-14).

In [ ]:
rv.plot_f401_fit();

## 11 · Scoreboard — every claim traceable

The final state of each problem, mirroring the paper and course tables. Sources: `datasets/dmir/results/nas-fronts/*.csv`, `datasets/dmir/docs/deployment.md` (measured 2026-07-13/14), `LOGBOOK.md`. The regression row against the internal reference is **not yet beaten**.

In [ ]:
from src import results_viz as rv

records = rv.records_table()
records.style.hide(axis="index")

## 12 · Export the baseline for STM32 (ONNX)

Exports the §6 baseline model to ONNX with a PyTorch↔ONNX-Runtime parity check. (The searched NAS models take a different path: Keras `.h5` → TFLite via `unas/prepare_deploy.py`, artifacts in `datasets/dmir/results/deploy/`.)

In [ ]:
import numpy as np
import onnxruntime as ort
import torch

EXPORT = True
if EXPORT:
    out_dir = ROOT / "models" / "exported"
    out_dir.mkdir(parents=True, exist_ok=True)
    onnx_path = out_dir / f"{cfg.run_name}-{cfg.task}.onnx"
    model.eval().cpu()
    example = torch.from_numpy(bundle.x_test[:1]).transpose(1, 2)
    torch.onnx.export(model, (example,), str(onnx_path), input_names=["window"],
                      output_names=["out"], dynamo=False)
    sess = ort.InferenceSession(str(onnx_path))
    ref = model(example).detach().numpy()
    got = sess.run(None, {"window": example.numpy()})[0]
    print(f"exported -> {onnx_path}")
    print(f"PyTorch/ONNX max abs diff: {np.abs(ref - got).max():.2e}")
    model.to(device)